# 3 · Schema matching  (per article)

For each **distinct OIE type** an article used, retrieve the nearest schema.org classes by
embedding similarity; every entity inherits its type's match. Matching is done at the type level
(the mention is deliberately not embedded), so the same type always resolves the same way.

Two document pools are compared, holding the query side fixed:
- **baseline** — the terse schema.org comments (`schema_classes.json`)
- **gemini**   — the Gemini-enriched definitions (`schema_classes_enriched.json`, built in nb 1)

and two embedding models (bge-small, e5-base). Set `ARTICLE_ID`, or run the final **batch** cell.
Canonical per-model outputs are written to
`results/<ARTICLE_ID>/schema_matching_<model>{,_by_type}.json` (from the **gemini** pool).

In [ ]:
# --- Config -------------------------------------------------------------------------------
import json, re
from pathlib import Path
import numpy as np

ARTICLE_ID  = "art1"                     # which article to match (or use the batch cell below)
SCHEMA_DIR  = Path("schema_org_files")   # shared pools (built by 1_schema_enrichment.ipynb)
RESULTS_DIR = Path("results")            # reads oie_entities.json, writes schema_matching_*.json

TOP_K = 5
ALIGN_QUERY_STYLE = True                 # append "Also called <synonyms>." to each OIE type query
                                         # (mirrors the enriched doc format; no-op without synonyms)

# Embedding models to compare. Each family needs its OWN prefix convention, applied separately to
# queries and documents. bge: query gets an instruction, documents get NO prefix. e5: query gets
# "query: ", documents get "passage: " (REQUIRED - e5 was trained this way; dropping the passage
# prefix badly hurts its scores).
EMBED_MODELS = ["BAAI/bge-small-en-v1.5", "intfloat/e5-base-v2"]

def embed_prefixes(model_name):
    n = model_name.lower()
    if "e5" in n:  return "query: ", "passage: "
    if "bge" in n: return "Represent this sentence for searching relevant passages: ", ""
    return "", ""   # gte / generic: no special prefixes

print("article:", ARTICLE_ID, "| embed:", EMBED_MODELS)

In [ ]:
# Run once to install dependencies (sentence-transformers pulls in torch, a large first download).
%pip install -q sentence-transformers numpy

## 1. Load the document pools + define retrieval

In [ ]:
# Document pools are shared across articles (built once by 1_schema_enrichment.ipynb).
# baseline = terse schema.org comments; gemini = Gemini-enriched definitions. Both use the same
# "Label (path): comment" embedding_text, so any difference is purely the enrichment.
baseline_pool = json.loads((SCHEMA_DIR / "schema_classes.json").read_text())
gemini_pool   = json.loads((SCHEMA_DIR / "schema_classes_enriched.json").read_text())
POOLS = {"baseline": baseline_pool, "gemini": gemini_pool}
print(f"{len(baseline_pool)} baseline classes | "
      f"{sum(c.get('enriched') for c in gemini_pool)}/{len(gemini_pool)} enriched")

from sentence_transformers import SentenceTransformer
_MODELS = {}   # load each embedding model once, reuse across articles
def get_model(name):
    if name not in _MODELS:
        _MODELS[name] = SentenceTransformer(name)
    return _MODELS[name]

def load_queries(article_id):
    """One query per DISTINCT OIE type = 'type: definition' (mention excluded), optionally with
    'Also called <synonyms>.' appended. Returns (oie, entities, oie_types, type_texts)."""
    oie = json.loads((RESULTS_DIR / article_id / "oie_entities.json").read_text())
    type_info = {t["type"]: t for t in oie.get("types", [])}
    entities  = oie["entities"]
    oie_types = list(dict.fromkeys(e["type"] for e in entities))
    def type_query_text(t):
        info = type_info.get(t, {})
        q = f"{t}: {info.get('definition', '')}".strip()
        if ALIGN_QUERY_STYLE and info.get("synonyms"):
            q = f"{q} Also called {', '.join(info['synonyms'])}."
        return q
    return oie, entities, oie_types, [type_query_text(t) for t in oie_types]

def retrieve(model, pool, type_texts, oie_types, qp, dp):
    """Embed queries + a document pool with the model's OWN prefixes; top-k per OIE type."""
    q_emb = np.asarray(model.encode([qp + t for t in type_texts],
                                    normalize_embeddings=True, batch_size=64))
    d_emb = np.asarray(model.encode([dp + c["embedding_text"] for c in pool],
                                    normalize_embeddings=True, batch_size=64, show_progress_bar=True))
    sims = q_emb @ d_emb.T
    topk = np.argsort(-sims, axis=1)[:, :TOP_K]
    return {t: [{"label": pool[j]["label"], "score": round(float(sims[ti, j]), 3),
                 "path": pool[j].get("path", ""),
                 "comment": pool[j].get("enriched_comment") or pool[j].get("comment", "")}
                for j in topk[ti]]
            for ti, t in enumerate(oie_types)}

def match_article(article_id):
    """Retrieve top-k for every (embedding model x pool). Returns (oie, entities, oie_types, results)
    where results[model][pool] = {oie_type: [candidates]}."""
    oie, entities, oie_types, type_texts = load_queries(article_id)
    results = {}
    for mname in EMBED_MODELS:
        qp, dp = embed_prefixes(mname)
        print(f"\n=== {mname}  query_prefix={qp!r} doc_prefix={dp!r} ===")
        m = get_model(mname)
        results[mname] = {pk: retrieve(m, pool, type_texts, oie_types, qp, dp)
                          for pk, pool in POOLS.items()}
    return oie, entities, oie_types, results

## 2. Match the selected article

In [ ]:
oie, entities, oie_types, results = match_article(ARTICLE_ID)
print("\nmodels:", list(results), "| pools:", list(POOLS))

## 3. Inspect results

In [ ]:
# Which model/pool to inspect in the per-type views below.
import statistics
SHOW_MODEL = EMBED_MODELS[0]     # embedding model shown in the per-type detail cells
SHOW_POOL  = "gemini"            # pool to inspect: "baseline" | "gemini"

def margin(cands):
    return cands[0]["score"] - cands[1]["score"] if len(cands) > 1 else float("nan")

# Mean top1-top2 margin for every (embedding model x pool). Higher = more decisive matches.
print("mean top1-top2 margin  (higher = more decisive)\n")
print(f"{'embedding model':<26}" + "".join(f"{pk:>12}" for pk in POOLS))
for mname in EMBED_MODELS:
    row = f"{mname:<26}"
    for pk in POOLS:
        row += f"{statistics.mean(margin(results[mname][pk][t]) for t in oie_types):>12.3f}"
    print(row)

In [ ]:
# Per-type top-1 for each embedding model (pool = SHOW_POOL). Format: label(score / margin).
print(f"top-1 schema match per OIE type   |   pool = {SHOW_POOL}\n")
print(f"{'OIE type':<34}" + "".join(f"{m.split('/')[-1]:<30}" for m in EMBED_MODELS))
for t in oie_types:
    row = f"{t:<34}"
    for mname in EMBED_MODELS:
        c = results[mname][SHOW_POOL][t]
        row += f"{c[0]['label']}({c[0]['score']:.2f}/m{margin(c):+.2f})".ljust(30)
    print(row)

In [ ]:
# Do the two embedding models agree on top-1? (per pool) + full top-k for the chosen model/pool.
if len(EMBED_MODELS) >= 2:
    a, b = EMBED_MODELS[0], EMBED_MODELS[1]
    sa, sb = a.split('/')[-1], b.split('/')[-1]
    print(f"top-1 agreement  {sa}  vs  {sb}:")
    for pk in POOLS:
        agree = sum(results[a][pk][t][0]["label"] == results[b][pk][t][0]["label"] for t in oie_types)
        print(f"  {pk:<10} {agree}/{len(oie_types)} = {agree/len(oie_types):.0%}")

print(f"\nfull top-{TOP_K}   model={SHOW_MODEL.split('/')[-1]}  pool={SHOW_POOL}")
for t in oie_types:
    print(f"\n>> {t}")
    for c in results[SHOW_MODEL][SHOW_POOL][t]:
        print(f"    {c['score']:.3f}  schema:{c['label']:<24} ({c['path']})")

## 4. Persist results

In [ ]:
# Persist the GEMINI-pool matches per embedding model into results/<id>/.
#   *_by_type.json : one entry per distinct OIE type (the actual matching unit)
#   *.json         : every entity with its type's match (handy for downstream joins)
def _slug(m): return m.split("/")[-1].replace(".", "_")

def persist(article_id, entities, results):
    out_dir = RESULTS_DIR / article_id
    out_dir.mkdir(parents=True, exist_ok=True)
    for mname in EMBED_MODELS:
        by_type = results[mname]["gemini"]
        (out_dir / f"schema_matching_{_slug(mname)}_by_type.json").write_text(
            json.dumps(by_type, indent=2, ensure_ascii=False))
        ent_rows = [{"entity": e["text"], "free_type": e["type"], "candidates": by_type[e["type"]]}
                    for e in entities]
        (out_dir / f"schema_matching_{_slug(mname)}.json").write_text(
            json.dumps(ent_rows, indent=2, ensure_ascii=False))
    print("wrote schema_matching_* ->", out_dir)

persist(ARTICLE_ID, entities, results)

## 5. Batch — match every article that has OIE entities

In [ ]:
# Runs matching for every results/<id>/oie_entities.json produced by notebook 2.
for fp in sorted(RESULTS_DIR.glob("*/oie_entities.json")):
    aid = fp.parent.name
    print("=" * 60, aid)
    _oie, _ents, _types, _res = match_article(aid)
    persist(aid, _ents, _res)
print("\ndone — all articles matched")